```markdown
# EDMS Integration Testing Guide

## Overview

This document provides comprehensive documentation for testing the EDMS (Enterprise Document Management System) integration between the web server and compute components. The system uses IPC (Inter-Process Communication) to spawn child processes for heavy compute operations.

## Architecture

```
┌─────────────┐     HTTP      ┌─────────────┐     IPC/Stdio    ┌─────────────┐
│   Client    │ ────────────> │ Web Server  │ ───────────────> │   Compute   │
│  (Browser)  │               │  (Port 3000) │                  │   Child     │
└─────────────┘               └─────────────┘                  └─────────────┘
                                    │                                  │
                                    │                                  │
                                    │     POST /internal/callback     │
                                    │ <───────────────────────────────│
                                    │                                  │
                                    ▼                                  ▼
                            ┌─────────────────────────────────────────┐
                            │         SQLite Database                 │
                            └─────────────────────────────────────────┘
```

## Prerequisites

### Required Tools
- Rust 1.70 or higher
- Cargo package manager
- SQLite3 development libraries
- WebSocket-capable client (for testing)

### Environment Setup

```bash
# Clone the repository
git clone <repository-url>
cd edms_with_tests

# Build all components
cargo build --workspace

# Build the compute child binary specifically
cd compute
cargo build --bin edms-child

# Build the web server
cd ../webserver
cargo build --bin rust-webserver
```

## Test Structure

```
compute/
├── tests/
│   ├── zipops_tests.rs          # Unit tests for zip operations
│   ├── ipc_child_test.rs        # IPC child process tests
│   ├── integration_tests.rs     # End-to-end integration tests
│   └── webserver_tests.rs       # Web server endpoint tests
├── src/
│   ├── zipops.rs                # ZIP file operations
│   ├── folder_manager.rs        # Folder structure management
│   └── bin/
│       └── child.rs             # IPC child process
└── Cargo.toml

webserver/
├── src/
│   ├── main.rs                  # Web server entry point
│   ├── handlers/                # Request handlers
│   └── ipc.rs                   # IPC communication
└── Cargo.toml
```

## Running Tests

### 1. Unit Tests

Unit tests focus on individual functions without external dependencies.

```bash
# Run all unit tests
cargo test --lib

# Run specific unit test
cargo test test_zip_folder

# Run with output visible
cargo test --lib -- --nocapture

# Run zipops tests specifically
cargo test --test zipops_tests -- --nocapture
```

### 2. Integration Tests

Integration tests require the web server to be running.

#### Option A: Manual Server Start

```bash
# Terminal 1: Start the web server
cd webserver
cargo run --bin rust-webserver

# Terminal 2: Run integration tests
cd compute
cargo test --test integration_tests -- --nocapture
```

#### Option B: Automated Test Script

```bash
#!/bin/bash
# test_with_server.sh

# Start server in background
cd webserver
cargo run --bin rust-webserver &
SERVER_PID=$!

# Wait for server to be ready
sleep 5

# Run tests
cd ../compute
cargo test --test integration_tests -- --nocapture

# Cleanup
kill $SERVER_PID
```

### 3. IPC Child Tests

These tests verify the child process functionality.

```bash
# Build the child binary first
cargo build --bin edms-child

# Run IPC tests
cargo test --test ipc_child_test -- --nocapture

# Run specific IPC test
cargo test test_child_export_collection -- --nocapture
```

### 4. WebSocket Tests

For testing real-time WebSocket connections.

```bash
# Run WebSocket tests
cargo test test_websocket_endpoints -- --nocapture

# With detailed logging
RUST_LOG=debug cargo test test_websocket_endpoints -- --nocapture
```

## Test Categories

### 1. ZIP Operations Tests (`zipops_tests.rs`)

Tests for file compression, extraction, and merging.

```rust
#[test]
fn test_zip_folder() -> DynResult<()> {
    let temp_dir = tempdir()?;
    let source = temp_dir.path().join("source");
    let output = temp_dir.path().join("output.zip");
    
    fs::create_dir_all(&source)?;
    create_test_file(&source, "test.txt", "content");
    
    zip_folder(&source, &output)?;
    
    assert!(output.exists());
    Ok(())
}
```

**What it tests:**
- Creating ZIP archives from directories
- Extracting ZIP files
- Merging multiple ZIP files
- SQLite database merging
- Path security validation

### 2. IPC Tests (`ipc_child_test.rs`)

Tests for the child process communication.

```rust
#[tokio::test]
async fn test_child_export_collection() -> TestResult {
    let temp_dir = tempdir()?;
    let source = temp_dir.path().join("source");
    let output = temp_dir.path().join("output.zip");
    
    // Create test files
    std::fs::create_dir_all(&source)?;
    std::fs::write(source.join("test.txt"), "test content")?;
    
    // Start mock callback server
    let mut callback_server = MockCallbackServer::start().await;
    
    // Prepare payload and spawn child
    let payload = json!({
        "source": source.to_string_lossy(),
        "output": output.to_string_lossy()
    });
    
    spawn_child_and_send_request("export_collection", payload, callback_server.port).await?;
    
    // Verify callback
    let callback = callback_server.wait_for_callback(5).await;
    assert!(callback.is_some());
    assert!(callback.unwrap().success);
    
    Ok(())
}
```

**What it tests:**
- Child process spawning
- IPC request/response flow
- Callback mechanism
- Error handling

### 3. Web Server Endpoint Tests

Tests for HTTP endpoints.

```rust
#[tokio::test]
async fn test_home_endpoint() {
    let client = Client::new();
    let response = client
        .get("http://localhost:3000/home")
        .send()
        .await
        .unwrap();
    
    assert_eq!(response.status(), 200);
    assert_eq!(response.text().await.unwrap(), "Ok");
}
```

**Available endpoints for testing:**
- `GET /home` - Health check
- `GET /list-view` - List view page
- `GET /dataview/dashboard` - Dashboard view
- `GET /dataview/:folder/active` - Active folder status
- `POST /internal/callback` - IPC callback endpoint
- WebSocket endpoints for real-time updates

## Debugging Tests

### Enable Logging

```bash
# Set log level
export RUST_LOG=debug

# Run test with logging
cargo test -- --nocapture --show-output

# For WebSocket debugging
export RUST_LOG=tokio_tungstenite=debug
```

### Backtrace on Failure

```bash
# Get full backtrace
RUST_BACKTRACE=1 cargo test

# Full backtrace with all frames
RUST_BACKTRACE=full cargo test
```

### Running Tests in Single Thread

```bash
# Avoid concurrency issues
cargo test -- --test-threads=1
```

### Filter Tests

```bash
# Run tests matching pattern
cargo test export_collection

# Run tests by module
cargo test zipops::

# Skip slow tests
cargo test -- --skip slow_test
```

## Continuous Integration

### GitHub Actions Workflow

```yaml
# .github/workflows/test.yml
name: Integration Tests

on: [push, pull_request]

jobs:
  test:
    runs-on: ubuntu-latest
    
    services:
      sqlite:
        image: sqlite:latest
        ports:
          - 5432:5432
    
    steps:
    - uses: actions/checkout@v3
    
    - name: Install Rust
      uses: actions-rs/toolchain@v1
      with:
        toolchain: stable
    
    - name: Build child binary
      run: |
        cd compute
        cargo build --bin edms-child
    
    - name: Start web server
      run: |
        cd webserver
        cargo run --bin rust-webserver &
        sleep 10
    
    - name: Run tests
      run: |
        cd compute
        cargo test -- --nocapture
    
    - name: Run integration tests
      run: |
        cd compute
        cargo test --test integration_tests -- --nocapture
```

## Common Test Issues and Solutions

### 1. Connection Refused

**Error:**
```
Connection refused (os error 111)
```

**Solution:**
```bash
# Ensure web server is running
curl http://localhost:3000/home

# If not running, start it
cd webserver
cargo run --bin rust-webserver
```

### 2. Child Binary Not Found

**Error:**
```
edms-child binary not found
```

**Solution:**
```bash
# Build the child binary
cd compute
cargo build --bin edms-child

# Verify it exists
ls target/debug/edms-child
```

### 3. Port Already in Use

**Error:**
```
Address already in use
```

**Solution:**
```bash
# Find process using port 3000
lsof -i :3000

# Kill the process
kill -9 <PID>

# Or use a different port
export PORT=3001
```

### 4. SQLite Version Conflicts

**Error:**
```
libsqlite3-sys conflicts with previous package
```

**Solution:**
```bash
# Clean and rebuild
cargo clean
cargo build --workspace
```

## Test Coverage

### Generate Coverage Report

```bash
# Install cargo-tarpaulin
cargo install cargo-tarpaulin

# Generate coverage
cargo tarpaulin --out Html

# Open report
open tarpaulin-report.html
```

### Coverage Targets

- **Unit Tests**: > 80% coverage
- **Integration Tests**: > 70% coverage
- **IPC Tests**: > 75% coverage

## Performance Testing

### Benchmark Tests

```rust
#[bench]
fn bench_zip_operation(b: &mut Bencher) {
    let temp_dir = tempdir().unwrap();
    let source = temp_dir.path().join("source");
    let output = temp_dir.path().join("output.zip");
    
    // Setup test data
    setup_large_directory(&source);
    
    b.iter(|| {
        zip_folder(&source, &output).unwrap();
    });
}
```

### Run Benchmarks

```bash
# Run benchmark tests
cargo bench

# Run specific benchmark
cargo bench -- zip_operation
```

## Manual Testing Script

```bash
#!/bin/bash
# manual_integration_test.sh

echo "EDMS Integration Test Suite"
echo "============================"

# Colors for output
GREEN='\033[0;32m'
RED='\033[0;31m'
NC='\033[0m' # No Color

# Test 1: Server Health
echo -n "Testing server health... "
curl -s http://localhost:3000/home > /dev/null
if [ $? -eq 0 ]; then
    echo -e "${GREEN}✓ PASSED${NC}"
else
    echo -e "${RED}✗ FAILED${NC}"
    exit 1
fi

# Test 2: Export Collection
echo -n "Testing export collection... "
mkdir -p /tmp/test/source
echo "test" > /tmp/test/source/test.txt

curl -s -X POST http://localhost:3000/api/export-collection \
  -H "Content-Type: application/json" \
  -d "{
    \"source\": \"/tmp/test/source\",
    \"output\": \"/tmp/test/output.zip\"
  }" > /dev/null

if [ -f "/tmp/test/output.zip" ]; then
    echo -e "${GREEN}✓ PASSED${NC}"
else
    echo -e "${RED}✗ FAILED${NC}"
fi

# Test 3: IPC Child Process
echo -n "Testing IPC child... "
echo '{"task":"export_collection","payload":{"source":"/tmp/test/source","output":"/tmp/test/child.zip"},"callback_port":3000}' | \
  ./compute/target/debug/edms-child 2>/dev/null

if [ -f "/tmp/test/child.zip" ]; then
    echo -e "${GREEN}✓ PASSED${NC}"
else
    echo -e "${RED}✗ FAILED${NC}"
fi

# Cleanup
rm -rf /tmp/test

echo -e "\n${GREEN}All tests completed${NC}"
```

## Best Practices

1. **Always clean before testing**
   ```bash
   cargo clean && cargo build
   ```

2. **Run tests in isolation**
   ```bash
   cargo test -- --test-threads=1
   ```

3. **Use temporary directories**
   ```rust
   let temp_dir = tempdir().unwrap();
   // Test automatically cleans up
   ```

4. **Add debug output for failing tests**
   ```rust
   println!("Response: {:?}", response);
   dbg!(variable);
   ```

5. **Test error cases as well**
   ```rust
   let result = function_that_should_fail();
   assert!(result.is_err());
   ```

## Troubleshooting Checklist

- [ ] Web server is running on port 3000
- [ ] Child binary is built (`cargo build --bin edms-child`)
- [ ] SQLite is installed (`sqlite3 --version`)
- [ ] No port conflicts (`lsof -i :3000`)
- [ ] Dependencies are up to date (`cargo update`)
- [ ] Build is clean (`cargo clean && cargo build`)
- [ ] Tests are running with `--nocapture` flag
- [ ] Logs are enabled for debugging

## Support

For issues or questions:
1. Check test output with `--nocapture`
2. Enable debug logging: `RUST_LOG=debug cargo test`
3. Run with backtrace: `RUST_BACKTRACE=1 cargo test`
4. Check the issue tracker for known problems
```

This documentation provides a comprehensive guide for testing the EDMS integration, including setup instructions, test categories, debugging techniques, and CI/CD integration.